# Loan Default Prediction System

## Objective
Build a machine learning workflow that predicts whether a loan applicant is likely to repay (`Y`) or default/not repay (`N`) using historical applicant data.

Because no external CSV was present in the workspace, this submission uses the included reproducible dataset `loan_data.csv`, which follows the requested schema.

## Phase 1 - Data Preprocessing
This phase loads the data, removes `Loan_ID`, handles missing values, converts categorical columns into numeric features, and scales numeric columns.

In [1]:
import math
from html import escape

import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    def display(value):
        print(value)


RANDOM_SEED = 42
DATA_PATH = "loan_data.csv"


def show_table(df, rows=5):
    """Display a compact table in notebooks and print it in plain Python."""
    if HTML is None:
        print(df.head(rows).to_string())
    else:
        display(HTML(df.head(rows).to_html(index=False)))


def display_svg(svg):
    """Render SVG in Jupyter; fall back to printing a short message otherwise."""
    if HTML is None:
        print("SVG chart generated.")
    else:
        display(HTML(svg))


def mode_value(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if len(mode) else None


def svg_bar(title, labels, values, width=720, height=330, color="#2f6f73"):
    """Create a simple SVG bar chart without external plotting libraries."""
    max_value = max(values) if len(values) else 1
    left = 70
    bottom = 55
    top = 45
    plot_width = width - left - 30
    plot_height = height - top - bottom
    gap = 16
    bar_width = (plot_width - gap * (len(values) - 1)) / max(len(values), 1)
    parts = [
        f'<svg viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="{escape(title)}">',
        f'<rect width="{width}" height="{height}" fill="#ffffff"/>',
        f'<text x="{width/2}" y="24" text-anchor="middle" font-size="18" font-family="Arial" fill="#1f2933">{escape(title)}</text>',
        f'<line x1="{left}" y1="{height-bottom}" x2="{width-20}" y2="{height-bottom}" stroke="#9aa5b1"/>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{height-bottom}" stroke="#9aa5b1"/>',
    ]
    for i, (label, value) in enumerate(zip(labels, values)):
        x = left + i * (bar_width + gap)
        bar_height = (value / max_value) * plot_height if max_value else 0
        y = height - bottom - bar_height
        parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" fill="{color}" rx="3"/>')
        parts.append(f'<text x="{x + bar_width/2:.1f}" y="{y - 7:.1f}" text-anchor="middle" font-size="12" font-family="Arial" fill="#1f2933">{value:.2f}</text>')
        parts.append(f'<text x="{x + bar_width/2:.1f}" y="{height - 28}" text-anchor="middle" font-size="12" font-family="Arial" fill="#1f2933">{escape(str(label))}</text>')
    parts.append("</svg>")
    return "\n".join(parts)


def svg_histogram(title, values, bins=12, width=720, height=330, color="#4d7cbd"):
    counts, edges = np.histogram(pd.Series(values).dropna(), bins=bins)
    labels = [f"{int(edges[i])}-{int(edges[i+1])}" for i in range(len(edges) - 1)]
    return svg_bar(title, labels, counts.astype(float).tolist(), width, height, color)


def svg_heatmap(title, corr, width=760, height=560):
    labels = list(corr.columns)
    cell_size = min((width - 180) / len(labels), (height - 130) / len(labels))
    left = 140
    top = 55
    parts = [
        f'<svg viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="{escape(title)}">',
        f'<rect width="{width}" height="{height}" fill="#ffffff"/>',
        f'<text x="{width/2}" y="24" text-anchor="middle" font-size="18" font-family="Arial" fill="#1f2933">{escape(title)}</text>',
    ]
    for i, row_label in enumerate(labels):
        parts.append(f'<text x="{left - 8}" y="{top + i*cell_size + cell_size*0.62:.1f}" text-anchor="end" font-size="11" font-family="Arial" fill="#1f2933">{escape(row_label)}</text>')
        parts.append(f'<text x="{left + i*cell_size + cell_size/2:.1f}" y="{top + len(labels)*cell_size + 18:.1f}" text-anchor="middle" font-size="11" font-family="Arial" fill="#1f2933">{escape(row_label)}</text>')
        for j, col_label in enumerate(labels):
            value = float(corr.iloc[i, j])
            if value >= 0:
                intensity = int(235 - 95 * min(value, 1))
                fill = f"rgb({intensity},{intensity + 10},255)"
            else:
                intensity = int(235 - 95 * min(abs(value), 1))
                fill = f"rgb(255,{intensity + 10},{intensity})"
            x = left + j * cell_size
            y = top + i * cell_size
            parts.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{cell_size:.1f}" height="{cell_size:.1f}" fill="{fill}" stroke="#ffffff"/>')
            parts.append(f'<text x="{x + cell_size/2:.1f}" y="{y + cell_size*0.58:.1f}" text-anchor="middle" font-size="10" font-family="Arial" fill="#1f2933">{value:.2f}</text>')
    parts.append("</svg>")
    return "\n".join(parts)


def prepare_features(df):
    """Remove ID, impute missing values, encode categoricals, and scale numerics."""
    cleaned = df.drop(columns=["Loan_ID"]).copy()
    y = cleaned["Loan_Status"].map({"N": 0, "Y": 1}).astype(int)
    X = cleaned.drop(columns=["Loan_Status"])

    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [column for column in X.columns if column not in numeric_cols]

    for column in numeric_cols:
        X[column] = X[column].fillna(X[column].median())
    for column in categorical_cols:
        X[column] = X[column].fillna(mode_value(X[column]))

    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=float)

    scaler = {}
    for column in numeric_cols:
        mean = X_encoded[column].mean()
        std = X_encoded[column].std(ddof=0)
        if std == 0:
            std = 1
        X_encoded[column] = (X_encoded[column] - mean) / std
        scaler[column] = {"mean": float(mean), "std": float(std)}

    return X_encoded.astype(float), y.astype(int), cleaned, scaler


def stratified_train_test_split(X, y, test_size=0.20, seed=RANDOM_SEED):
    """Create an 80/20 split while keeping the target distribution similar."""
    rng = np.random.default_rng(seed)
    train_indices = []
    test_indices = []

    for class_value in sorted(y.unique()):
        class_indices = y[y == class_value].index.to_numpy().copy()
        rng.shuffle(class_indices)
        n_test = int(round(len(class_indices) * test_size))
        test_indices.extend(class_indices[:n_test])
        train_indices.extend(class_indices[n_test:])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)

    return (
        X.loc[train_indices].to_numpy(dtype=float),
        X.loc[test_indices].to_numpy(dtype=float),
        y.loc[train_indices].to_numpy(dtype=int),
        y.loc[test_indices].to_numpy(dtype=int),
        X.columns.tolist(),
    )


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -35, 35)))


def train_logistic_regression(X_train, y_train, learning_rate=0.08, epochs=2500):
    X_bias = np.c_[np.ones(X_train.shape[0]), X_train]
    weights = np.zeros(X_bias.shape[1])

    for _ in range(epochs):
        predictions = sigmoid(X_bias @ weights)
        gradient = X_bias.T @ (predictions - y_train) / len(y_train)
        weights -= learning_rate * gradient

    return weights


def predict_logistic_regression(weights, X):
    X_bias = np.c_[np.ones(X.shape[0]), X]
    probabilities = sigmoid(X_bias @ weights)
    return (probabilities >= 0.5).astype(int), probabilities


def gini_impurity(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    return 1 - p ** 2 - (1 - p) ** 2


class DecisionTreeClassifierScratch:
    """Small Gini-based decision tree classifier for numeric encoded features."""

    def __init__(self, max_depth=4, min_samples_split=18, max_features=None, seed=RANDOM_SEED):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.rng = np.random.default_rng(seed)
        self.tree_ = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.tree_ = self._build_tree(X, y, depth=0)
        return self

    def _candidate_features(self):
        if self.max_features is None:
            return np.arange(self.n_features_)
        count = min(self.max_features, self.n_features_)
        return self.rng.choice(self.n_features_, size=count, replace=False)

    def _best_split(self, X, y):
        parent_impurity = gini_impurity(y)
        best_gain = 0
        best_feature = None
        best_threshold = None

        for feature in self._candidate_features():
            values = np.unique(X[:, feature])
            if len(values) <= 1:
                continue
            thresholds = (values[:-1] + values[1:]) / 2

            if len(thresholds) > 25:
                quantiles = np.linspace(0.05, 0.95, 25)
                thresholds = np.unique(np.quantile(values, quantiles))

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                weighted_impurity = (
                    left_mask.mean() * gini_impurity(y[left_mask])
                    + right_mask.mean() * gini_impurity(y[right_mask])
                )
                gain = parent_impurity - weighted_impurity
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _build_tree(self, X, y, depth):
        positive_rate = float(np.mean(y))
        prediction = int(positive_rate >= 0.5)

        if (
            depth >= self.max_depth
            or len(y) < self.min_samples_split
            or len(np.unique(y)) == 1
        ):
            return {"prediction": prediction, "positive_rate": positive_rate}

        feature, threshold, gain = self._best_split(X, y)
        if feature is None or gain <= 1e-7:
            return {"prediction": prediction, "positive_rate": positive_rate}

        left_mask = X[:, feature] <= threshold
        return {
            "feature": int(feature),
            "threshold": float(threshold),
            "prediction": prediction,
            "positive_rate": positive_rate,
            "left": self._build_tree(X[left_mask], y[left_mask], depth + 1),
            "right": self._build_tree(X[~left_mask], y[~left_mask], depth + 1),
        }

    def _predict_one(self, row, node):
        while "feature" in node:
            if row[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["prediction"], node["positive_rate"]

    def predict(self, X):
        predictions = [self._predict_one(row, self.tree_)[0] for row in X]
        return np.array(predictions, dtype=int)

    def predict_proba(self, X):
        probabilities = [self._predict_one(row, self.tree_)[1] for row in X]
        return np.array(probabilities, dtype=float)


class RandomForestClassifierScratch:
    """Bagged decision trees with random feature subsets at each split."""

    def __init__(self, n_estimators=35, max_depth=5, min_samples_split=16, seed=RANDOM_SEED):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.rng = np.random.default_rng(seed)
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        max_features = max(1, int(math.sqrt(X.shape[1])))
        for i in range(self.n_estimators):
            sample_indices = self.rng.choice(len(X), size=len(X), replace=True)
            tree = DecisionTreeClassifierScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_features,
                seed=RANDOM_SEED + i + 1,
            )
            tree.fit(X[sample_indices], y[sample_indices])
            self.trees.append(tree)
        return self

    def predict_proba(self, X):
        probabilities = np.vstack([tree.predict_proba(X) for tree in self.trees])
        return probabilities.mean(axis=0)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


def classification_metrics(y_true, y_pred):
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
show_table(df)

Dataset shape: (614, 13)


Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
LP100001,Male,Yes,3+,Graduate,No,6155,0,175.0,360.0,1.0,Urban,Y
LP100002,Male,No,0,Not Graduate,No,3311,4109,218.0,360.0,1.0,Rural,Y
LP100003,Female,No,0,Graduate,No,1730,1753,144.0,360.0,1.0,Rural,Y
LP100004,Male,Yes,1,Graduate,No,6956,0,194.0,480.0,1.0,Rural,Y
LP100005,Male,Yes,0,Graduate,Yes,16582,0,374.0,360.0,1.0,Rural,N


In [3]:
missing_summary = df.isna().sum().reset_index()
missing_summary.columns = ["Column", "Missing Values"]
missing_summary

,Column,Missing Values
0,Loan_ID,0
1,Gender,9
2,Married,13
3,Dependents,29
4,Education,0
5,Self_Employed,17
6,ApplicantIncome,0
7,CoapplicantIncome,0
8,LoanAmount,21
9,Loan_Amount_Term,14


In [4]:
X, y, cleaned_df, scaler = prepare_features(df)
print(f"Feature matrix shape after preprocessing: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True).rename({0: 'N', 1: 'Y'}).round(3)}")
show_table(X)

Feature matrix shape after preprocessing: (614, 14)
Target distribution:
Loan_Status
Y    0.702
N    0.298
Name: proportion, dtype: float64


ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Gender_Male,Married_Yes,Dependents_1,Dependents_2,Dependents_3+,Education_Not Graduate,Self_Employed_Yes,Property_Area_Semiurban,Property_Area_Urban
0.042794,-0.833167,-0.162853,0.243643,0.435801,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
-0.753373,1.059382,0.325577,0.243643,0.435801,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
-1.195968,-0.025759,-0.514977,0.243643,0.435801,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.267031,-0.833167,0.052965,2.176422,0.435801,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2.961793,-0.833167,2.097554,0.243643,0.435801,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Phase 2 - Exploratory Data Analysis
The charts below show distribution patterns, feature-target comparisons, and numeric correlations.

In [5]:
status_counts = df["Loan_Status"].value_counts().reindex(["N", "Y"]).fillna(0)
display_svg(svg_bar("Loan Status Counts", status_counts.index.tolist(), status_counts.astype(float).tolist()))

In [6]:
display_svg(svg_histogram("Applicant Income Distribution", df["ApplicantIncome"], bins=12))
display_svg(svg_histogram("Loan Amount Distribution", df["LoanAmount"], bins=12, color="#8a6f3d"))

In [7]:
approval_by_credit = (
    df.assign(Credit_History=df["Credit_History"].fillna(-1).replace({-1: "Missing"}))
    .groupby("Credit_History")["Loan_Status"]
    .apply(lambda series: (series == "Y").mean())
    .reset_index()
)
display_svg(svg_bar(
    "Repayment Rate by Credit History",
    approval_by_credit["Credit_History"].astype(str).tolist(),
    approval_by_credit["Loan_Status"].round(3).tolist(),
    color="#7d5a9e",
))

In [8]:
approval_by_property = (
    df.groupby("Property_Area")["Loan_Status"]
    .apply(lambda series: (series == "Y").mean())
    .sort_values(ascending=False)
)
display_svg(svg_bar(
    "Repayment Rate by Property Area",
    approval_by_property.index.tolist(),
    approval_by_property.round(3).tolist(),
    color="#2f855a",
))

In [9]:
corr_df = df.drop(columns=["Loan_ID"]).copy()
corr_df["Loan_Status_Num"] = corr_df["Loan_Status"].map({"N": 0, "Y": 1})
numeric_corr = corr_df.select_dtypes(include=[np.number]).corr(numeric_only=True).round(2)
display_svg(svg_heatmap("Numeric Correlation Matrix", numeric_corr))
numeric_corr

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status_Num
ApplicantIncome,1.00,0.07,0.86,-0.04,0.01,0.06
CoapplicantIncome,0.07,1.00,0.42,-0.04,0.06,0.09
LoanAmount,0.86,0.42,1.00,-0.06,0.04,0.05
Loan_Amount_Term,-0.04,-0.04,-0.06,1.00,0.01,0.10
Credit_History,0.01,0.06,0.04,0.01,1.00,0.37
Loan_Status_Num,0.06,0.09,0.05,0.10,0.37,1.00


## Phase 3 - Model Building
We use an 80/20 stratified split and train three classifiers:

- Logistic Regression
- Decision Tree
- Random Forest-style bagged trees

In [10]:
X_train, X_test, y_train, y_test, feature_names = stratified_train_test_split(X, y, test_size=0.20)
print(f"Training records: {len(y_train)}")
print(f"Testing records: {len(y_test)}")

Training records: 491
Testing records: 123


In [11]:
logistic_weights = train_logistic_regression(X_train, y_train)
logistic_pred, logistic_proba = predict_logistic_regression(logistic_weights, X_test)

decision_tree = DecisionTreeClassifierScratch(max_depth=4, min_samples_split=18)
decision_tree.fit(X_train, y_train)
tree_pred = decision_tree.predict(X_test)

random_forest = RandomForestClassifierScratch(n_estimators=35, max_depth=5, min_samples_split=16)
random_forest.fit(X_train, y_train)
forest_pred = random_forest.predict(X_test)

results = pd.DataFrame([
    {"Model": "Logistic Regression", **classification_metrics(y_test, logistic_pred)},
    {"Model": "Decision Tree", **classification_metrics(y_test, tree_pred)},
    {"Model": "Random Forest", **classification_metrics(y_test, forest_pred)},
]).sort_values(["F1-Score", "Accuracy"], ascending=False)

results_display = results.copy()
for metric in ["Accuracy", "Precision", "Recall", "F1-Score"]:
    results_display[metric] = results_display[metric].round(3)
results_display

,Model,Accuracy,Precision,Recall,F1-Score,TN,FP,FN,TP
1,Decision Tree,0.740,0.765,0.907,0.830,13,24,8,78
2,Random Forest,0.715,0.752,0.884,0.813,12,25,10,76
0,Logistic Regression,0.699,0.747,0.860,0.800,12,25,12,74


In [12]:
best_model = results.iloc[0]
print(f"Best model: {best_model['Model']}")
print(f"Accuracy: {best_model['Accuracy']:.3f}")
print(f"Precision: {best_model['Precision']:.3f}")
print(f"Recall: {best_model['Recall']:.3f}")
print(f"F1-Score: {best_model['F1-Score']:.3f}")

Best model: Decision Tree
Accuracy: 0.740
Precision: 0.765
Recall: 0.907
F1-Score: 0.830


## Phase 4 - Model Comparison
The best model for this run is **Decision Tree**, selected by F1-score and accuracy. It achieved **0.740 accuracy** and **0.830 F1-score** on the held-out test set.

## Phase 5 - Findings
- Credit history is the strongest predictor of loan repayment.
- Semiurban/urban property areas have higher repayment rates than rural areas in this dataset.
- Income and loan amount are right-skewed, so median imputation and scaling are useful preprocessing steps.
- The final model can support loan-screening decisions, but it should be validated on real institutional data before production use.